# Lesson 39 · Writing the FFT from scratch — rebuilding the 1965 algorithm that changed the world: a recursive Cooley–Tukey FFT, error < 1e-10

**Goal**: first string the even/odd split, the recursive base case and the butterfly merge together into one line of thought, then write the recursive Cooley–Tukey FFT yourself and squeeze the error below $10^{-10}$.

> **What matters in this implementation lesson**: the recursive base case is $N=1$ (a one-point DFT is the point itself). L38 showed you *why* it's fast; this lesson you write the recursion. Bit reversal can wait — it's optional, for later.

🔗 **Aurora connection**: put it side by side with `fft()` in `transforms.py` and you'll see that the recursive version you write here and the production code are mathematically identical.

← **Previous lesson** · [L38 · The FFT butterfly](L38_fft_butterfly.ipynb)

> Last lesson covered **the FFT butterfly**: the even/odd split, the butterfly E[k]+W^k·O[k], O(N²)→O(N log N).
> This lesson explores **writing the FFT from scratch**.

## This lesson's story: why does "cutting a big problem in half" make it 200 times faster?

There's a classic interview puzzle: in a tower with 1024 floors, finding one particular floor takes nothing more than binary search — halve the range each time, and you need at most log₂1024 = 10 steps.

The FFT does exactly the same thing. The DFT scans all N samples for every frequency bin — N bins × N samples = **N² multiplications**. The FFT asks: "Could we scan only half the samples, then merge two half results into the full result?"

**Cooley and Tukey's answer** (1965):

$$X[k] = \underbrace{\text{DFT}(x_{\text{even}})[k]}_{E[k]} + e^{-2\pi i k/N} \cdot \underbrace{\text{DFT}(x_{\text{odd}})[k]}_{O[k]}$$

Take an N/2-point DFT of the "even-index sequence" and another of the "odd-index sequence", then one butterfly merge gives you the full N-point result. This divide and conquer **can recurse**: N/2 → N/4 → ... → 1 (base case: a one-point DFT is just x[0]).

**Efficiency**: log₂N stages × N/2 butterflies per stage = **(N/2)·log₂N** multiplications.

| N | DFT multiplications | FFT multiplications | Speedup |
|---|---|---|---|
| 64 | 4096 | 192 | 21× |
| 1024 | 1048576 | 5120 | 205× |
| 4096 | 16777216 | 24576 | 682× |

Your task in this section: implement `my_fft(x)`, a recursive Cooley–Tukey FFT, and verify that its error against `np.fft.fft` is < 1e-10.

## Groundwork: deriving the Cooley–Tukey formula from the DFT definition

Students ask, "Where does this formula come from?" — so let's start from the plain DFT definition and derive, step by step, why an even/odd split makes things faster.

**The DFT definition** (you know it well by now):
$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-2\pi i k n / N}$$

Now split the sum in two: the even indices $n=2m$ and the odd indices $n=2m+1$:
$$X[k] = \sum_{m=0}^{N/2-1} x[2m] \cdot e^{-2\pi i k (2m) / N} + \sum_{m=0}^{N/2-1} x[2m+1] \cdot e^{-2\pi i k (2m+1) / N}$$

In the first sum, the exponent becomes $e^{-2\pi i k m / (N/2)}$ — exactly the form of **a DFT of length N/2**. Call it $E[k] = \text{DFT}(x[0,2,4,...])$.

In the second sum, first pull out $e^{-2\pi i k / N}$ (this is the "twiddle factor"); what's left is $e^{-2\pi i k m / (N/2)}$, which is $\text{DFT}(x[1,3,5,...])$. Call it $O[k] = \text{DFT}(x[1,3,5,...])$.

**So**:
$$X[k] = E[k] + e^{-2\pi i k/N} \cdot O[k]$$

Now for the partner frequency $X[k + N/2]$. By periodicity, $e^{-2\pi i (k+N/2) n / N} = e^{-2\pi i k n / N} \cdot e^{-2\pi i n/2}$, and $e^{-2\pi i n/2}$ gives $e^{-2\pi i m} = 1$ for even $n=2m$ (a whole number of turns), and $e^{-\pi i} \cdot e^{-2\pi i m} = -1$ for odd $n=2m+1$. So the even sum stays exactly as it was, the odd sum flips sign, and:

$$X[k+N/2] = E[k] - e^{-2\pi i k/N} \cdot O[k]$$

**That's the complete derivation of the Cooley–Tukey formula.** The minus sign doesn't come from nowhere — the periodicity of the complex exponential hands it to us for free.

## Key concept: complex multiplication is rotation — why $e^{-2\pi i k/N}$ is a "twiddle factor"

You might see the code `twiddle = np.exp(-2j * np.pi * k / N)` followed by `E + twiddle * O` and think it's "just a complex multiplication". But **why does that multiplication stand for a "rotation"**? Let's build the geometric intuition.

**Euler's formula, revisited**:
$$e^{i\theta} = \cos\theta + i\sin\theta$$

What does this expression represent on the complex plane? **A point on the unit circle** at angle $\theta$ (measured counterclockwise from the positive real axis).

- $e^{i \cdot 0} = 1 + 0i = (1, 0)$ — pointing east
- $e^{i \pi/2} = 0 + 1i = (0, 1)$ — pointing north
- $e^{i \pi} = -1 + 0i = (-1, 0)$ — pointing west
- $e^{i \cdot 3\pi/2} = 0 - 1i = (0, -1)$ — pointing south

**Complex multiplication is rotation**: when you multiply another complex number $z = a + bi$ by $e^{i\theta}$, the result is "$z$ rotated about the origin by the angle $\theta$". For example:

$$e^{i\pi/2} \cdot 1 = i$$

On the complex plane, 1 is the unit vector pointing east; multiply it by $e^{i\pi/2}$ ("rotate by 90°") and you get $i$, pointing north.

$$e^{i\pi} \cdot (2+0i) = (-1) \cdot (2+0i) = -2$$

Multiply 2 by $e^{i\pi}$ (rotate by 180°) and it points west.

**In the FFT**: the twiddle factor $e^{-2\pi i k/N}$ rotates by a different angle for each frequency bin $k=0,1,2,...,N/2-1$ ($0° \to -\frac{360°}{N} \to -\frac{720°}{N} \to ...$). These rotations "line up" the phase difference between the even terms and the odd terms — because, in the original signal, the odd positions naturally lag the even positions by one sample (and a time delay is a phase rotation).

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
# Demo: complex multiplication is rotation
import numpy as np
import matplotlib.pyplot as plt

# Example 1: (1+1j) times e^(iπ/4) = a 45° rotation
z1 = 1 + 1j
angle1 = np.pi / 4
rot1 = np.exp(1j * angle1)
z1_rotated = rot1 * z1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))

# left plot: a 45° rotation
ax1.arrow(0, 0, z1.real, z1.imag, head_width=0.1, head_length=0.08, fc='blue', ec='blue', alpha=0.7, label='z = 1+1j')
ax1.arrow(0, 0, z1_rotated.real, z1_rotated.imag, head_width=0.1, head_length=0.08, fc='red', ec='red', alpha=0.7, label=f'z × e^(iπ/4) [rotated 45°]')
circle = plt.Circle((0, 0), np.abs(z1), fill=False, linestyle='--', color='gray', alpha=0.5)
ax1.add_patch(circle)
ax1.set_xlim(-2, 2); ax1.set_ylim(-2, 2)
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)
ax1.set_xlabel('Real'); ax1.set_ylabel('Imag')
ax1.set_title('Complex multiplication = rotation (same length, angle increases)')
ax1.legend()

# right plot: the twiddle factors stepping round
N = 8
k_vals = np.arange(N // 2)
twiddle_vals = np.exp(-2j * np.pi * k_vals / N)
colors = plt.cm.rainbow(np.linspace(0, 1, len(twiddle_vals)))
for k, tw, col in zip(k_vals, twiddle_vals, colors):
    angle_deg = -360 * k / N
    ax2.arrow(0, 0, tw.real, tw.imag, head_width=0.08, head_length=0.06, fc=col, ec=col, alpha=0.8)
    ax2.text(tw.real*1.3, tw.imag*1.3, f'k={k}\n({angle_deg:.0f}°)', fontsize=8, ha='center')
circle = plt.Circle((0, 0), 1, fill=False, linestyle='--', color='gray', alpha=0.5)
ax2.add_patch(circle)
ax2.set_xlim(-1.5, 1.5); ax2.set_ylim(-1.5, 1.5)
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)
ax2.set_xlabel('Real'); ax2.set_ylabel('Imag')
ax2.set_title(f'FFT twiddle factors N={N}: e^(-2πik/N), k=0..{N//2-1}')

plt.tight_layout()
plt.show()

# numeric demo
print("Complex multiplication by hand: (1+0j) × e^(-iπ/2)")
a = 1 + 0j
b = np.exp(-1j * np.pi / 2)
result = a * b
print(f"  e^(-iπ/2) = {b:.4f}  (should be -i = 0-1j)")
print(f"  (1+0j) × {b:.4f} = {result:.4f}  (rotated -90°, from east round to south)")
print()

print("By hand: x_odd times the twiddle factor, the N=4, k=1 case")
O_k1 = 2 + 0j  # suppose O[1] = 2+0j
twiddle_4_1 = np.exp(-2j * np.pi * 1 / 4)
result = O_k1 * twiddle_4_1
print(f"  O[1] = {O_k1}")
print(f"  W_4^1 = e^(-2πi/4) = e^(-πi/2) = {twiddle_4_1:.4f}")
print(f"  O[1] × W_4^1 = {O_k1} × {twiddle_4_1:.4f} = {result:.4f}  (rotated -90°)")


## Worked examples by hand: the FFT for N=2 and N=4

Now let's walk through the Cooley–Tukey formula with real numbers, so you can watch the "split → merge" process happen value by value.

### Example 1: N=2, input $x = [1, 2]$

For a length-2 input, following Cooley–Tukey:

**Even/odd split**:
- $x_{\text{even}} = x[0] = [1]$  (just one element, length = 1)
- $x_{\text{odd}} = x[1] = [2]$  (just one element, length = 1)

**Base case** (return directly when $N=1$):
- $E = \text{FFT}([1]) = [1]$  (for a length-1 signal, the DFT is the signal itself)
- $O = \text{FFT}([2]) = [2]$

**Twiddle factors** (k runs over 0..N/2-1 = 0..0, so there's only one value):
- $k=0$: $W_2^0 = e^{-2\pi i \cdot 0 / 2} = e^0 = 1$

**Butterfly merge**:
- $X[0] = E[0] + W_2^0 \cdot O[0] = 1 + 1 \cdot 2 = 3$
- $X[1] = E[0] - W_2^0 \cdot O[0] = 1 - 1 \cdot 2 = -1$

**Answer**: $X = [3, -1]$

**Check** (with the DFT definition):
- $X[0] = x[0] \cdot e^0 + x[1] \cdot e^0 = 1 + 2 = 3$ ✓
- $X[1] = x[0] \cdot e^0 + x[1] \cdot e^{-\pi i} = 1 + 2 \cdot (-1) = -1$ ✓

### Example 2: N=4, input $x = [1, 0, -1, 0]$ (a student whiteboard problem)

**Level 1: split**
- $x_{\text{even}} = x[0,2] = [1, -1]$
- $x_{\text{odd}} = x[1,3] = [0, 0]$

**Level 2: recurse down to N=2**

For $x_{\text{even}} = [1, -1]$, use the method from the example above (with different numbers):
- E's even part = $[1]$, E's odd part = $[-1]$
- twiddle factor: $W_2^0 = 1$
- $E = [\underbrace{1 + (-1) \cdot 1}_{0}, \underbrace{1 - (-1) \cdot 1}_{2}] = [0, 2]$

For $x_{\text{odd}} = [0, 0]$:
- O's even part = $[0]$, O's odd part = $[0]$
- twiddle factor: $W_2^0 = 1$
- $O = [0 + 0 \cdot 1, 0 - 0 \cdot 1] = [0, 0]$

**Level 1: merge E and O**

Now we have $E = [0, 2]$ and $O = [0, 0]$, and we use the twiddle factors to merge them into a length-4 $X$.

- $k$ runs over 0..1 ($N/2 - 1$)
- $k=0$: $W_4^0 = e^{-2\pi i \cdot 0 / 4} = 1$
  - $X[0] = E[0] + 1 \cdot O[0] = 0 + 0 = 0$
  - $X[2] = E[0] - 1 \cdot O[0] = 0 - 0 = 0$
- $k=1$: $W_4^1 = e^{-2\pi i / 4} = e^{-i\pi/2} = -i$
  - $X[1] = E[1] + (-i) \cdot O[1] = 2 + (-i) \cdot 0 = 2$
  - $X[3] = E[1] - (-i) \cdot O[1] = 2 - (-i) \cdot 0 = 2$

**Answer**: $X = [0, 2, 0, 2]$

**Check** (with the DFT definition):
- $X[0] = 1 + 0 - 1 + 0 = 0$ ✓
- $X[1] = 1 \cdot e^0 + 0 \cdot e^{-i\pi/2} + (-1) \cdot e^{-i\pi} + 0 \cdot e^{-3i\pi/2}$
  $= 1 + 0 + (-1) \cdot (-1) + 0 = 1 + 1 = 2$ ✓

Now you've seen the concrete process: "solve the small problems (N=2) recursively, then merge back up level by level".

In [ ]:
# Check the two hand-worked examples above
import numpy as np

print("=" * 60)
print("Example 1 check: N=2, x=[1, 2]")
print("=" * 60)
x1 = np.array([1, 2], dtype=complex)
X1_manual = np.array([3, -1], dtype=complex)
X1_numpy = np.fft.fft(x1)
print(f"by hand:   X = {X1_manual}")
print(f"numpy.fft: X = {X1_numpy}")
print(f"same? {np.allclose(X1_manual, X1_numpy)}")
print()

print("=" * 60)
print("Example 2 check: N=4, x=[1, 0, -1, 0]")
print("=" * 60)
x2 = np.array([1, 0, -1, 0], dtype=complex)
X2_manual = np.array([0, 2, 0, 2], dtype=complex)
X2_numpy = np.fft.fft(x2)
print(f"by hand:   X = {X2_manual}")
print(f"numpy.fft: X = {np.round(X2_numpy, 4)}")
print(f"same? {np.allclose(X2_manual, X2_numpy)}")
print()

# Break example 2 down into its intermediate steps
print("=" * 60)
print("Example 2 in detail: the recursion step by step")
print("=" * 60)
x2 = np.array([1, 0, -1, 0], dtype=complex)
x_even = x2[::2]  # [1, -1]
x_odd = x2[1::2]  # [0, 0]
print(f"original x = {x2}")
print(f"  even positions x_even = x[::2] = {x_even}")
print(f"  odd positions x_odd = x[1::2] = {x_odd}")
print()

E = np.fft.fft(x_even)
O = np.fft.fft(x_odd)
print(f"recursive FFT (even positions): E = FFT({x_even}) = {np.round(E, 4)}")
print(f"recursive FFT (odd positions): O = FFT({x_odd}) = {np.round(O, 4)}")
print()

N = len(x2)
k = np.arange(N // 2)
twiddle = np.exp(-2j * np.pi * k / N)
print(f"twiddle factors k={k}:")
for ki, tw in zip(k, twiddle):
    print(f"  W_4^{ki} = e^(-2πi·{ki}/4) = {tw:.4f}")
print()

top = E + twiddle * O
bottom = E - twiddle * O
X_reconstructed = np.concatenate([top, bottom])
print(f"butterfly merge:")
print(f"  X[0..1] = E + twiddle*O = {np.round(E, 4)} + {np.round(twiddle*O, 4)} = {np.round(top, 4)}")
print(f"  X[2..3] = E - twiddle*O = {np.round(E, 4)} - {np.round(twiddle*O, 4)} = {np.round(bottom, 4)}")
print(f"stitched result: X = {np.round(X_reconstructed, 4)}")
print(f"numpy.fft result: {np.round(np.fft.fft(x2), 4)}")
print(f"same? {np.allclose(X_reconstructed, np.fft.fft(x2))}")


## Quick review: the NumPy basics the code will use

If all you've learned so far is Python's `print`, variables, `for` loops and lists, here are the three key NumPy operations that appear in `my_fft`.

### 1. `np.arange(N // 2)` — making an array of integers

```python
k = np.arange(4)  # makes the array [0, 1, 2, 3]
k = np.arange(3)  # makes the array [0, 1, 2]
k = np.arange(4 // 2)  # makes the array [0, 1] (integer division 4//2=2)
```

This generates all the k values "in one batch" instead of one by one in a `for` loop.

### 2. Array slicing `x[::2]` and `x[1::2]` — pulling out the even/odd indices

```python
x = np.array([1, 2, 3, 4, 5, 6])
x[::2]   # every other element, starting at 0: [1, 3, 5]
x[1::2]  # every other element, starting at 1: [2, 4, 6]
```

In the FFT, `x[::2]` gives the even positions and `x[1::2]` the odd positions.

### 3. Vectorized multiplication `twiddle * O` — element by element

```python
twiddle = np.array([1+0j, -0.7+0.7j])  # two complex numbers
O = np.array([2+0j, 3+0j])  # two complex numbers
result = twiddle * O  # element by element: [(1+0j)*(2+0j), (-0.7+0.7j)*(3+0j)]
                      # result: [2+0j, -2.1+2.1j]
```

This is a "vectorized" operation — one line of code multiplies every element of the array, with no `for` loop needed.

In [ ]:
# Demo of the basic NumPy operations
import numpy as np

print("Operation 1: np.arange(N // 2)")
print("-" * 50)
for N in [4, 8, 16]:
    k = np.arange(N // 2)
    print(f"  N={N}:  np.arange({N}//2) = np.arange({N//2}) = {k}")
print()

print("Operation 2: array slicing x[::2] and x[1::2]")
print("-" * 50)
x = np.array([10, 20, 30, 40, 50, 60, 70, 80])
print(f"  x = {x}")
print(f"  x[::2]   (even indices) = {x[::2]}")
print(f"  x[1::2]  (odd indices) = {x[1::2]}")
print()

print("Operation 3: vectorized complex multiplication")
print("-" * 50)
# simulate the N=4 case
N = 4
k = np.arange(N // 2)
twiddle = np.exp(-2j * np.pi * k / N)
O = np.array([2+0j, 1+1j])  # pretend this came back from one of the recursive calls

print(f"  N={N}")
print(f"  k = {k}")
print(f"  twiddle = np.exp(-2j*π*k/N) = {np.round(twiddle, 4)}")
print(f"  O = {O}")
print(f"  twiddle * O = {np.round(twiddle * O, 4)}")
print()
print("  ✓ this is the 'twiddle * O' part of the butterfly formula")


## A close-up look: the recursion step by step (with real numbers)

"Recursion" sounds abstract. Let's take a real N=4 example and show, step by step, **the order in which the program actually runs** and **the values at every step**.

### Execution trace: everything `my_fft([1, 0, -1, 0])` does

```
Call stack                          Return value
─────────────────────────────────────────────────────────
1️⃣  my_fft([1,0,-1,0], N=4)        entered, not returned yet
    ├─ E = my_fft([1,-1], N=2)      recursive call on the even part, entered
    │  ├─ E = my_fft([1], N=1)      entered, hits bottom
    │  │  return [1]              ◀─ returns: base case
    │  ├─ O = my_fft([-1], N=1)     entered, hits bottom
    │  │  return [-1]             ◀─ returns: base case
    │  ├─ butterfly merge E=[1], O=[-1]
    │  │   k=0: W_2^0 = 1
    │  │   top    = E + twiddle*O = [1] + 1*[-1] = [0]
    │  │   bottom = E - twiddle*O = [1] - 1*[-1] = [2]
    │  return [0, 2]              ◀─ returns: E done
    ├─ O = my_fft([0,0], N=2)       recursive call on the odd part, entered
    │  ├─ my_fft([0], N=1) → [0]
    │  ├─ my_fft([0], N=1) → [0]
    │  ├─ butterfly merge
    │  return [0, 0]              ◀─ returns: O done
    ├─ E=[0,2], O=[0,0], now merge them
    │  k=[0,1]:
    │    W_4^0 = e^0 = 1
    │    W_4^1 = e^(-iπ/2) = -i
    │  top    = [0,2] + [1,-i]*[0,0] = [0,2] + [0,0] = [0,2]
    │  bottom = [0,2] - [1,-i]*[0,0] = [0,2] - [0,0] = [0,2]
    return [0,2,0,2]             ◀─ returns: FFT complete
```

Key observations:
- **Depth first**: only after `my_fft([1,-1], N=2)` has completely returned does the computation of `my_fft([0,0], N=2)` begin
- **The base case is what makes it work**: when N=1, the input is returned directly (the first `return [1]` in the trace), and that is what lets the whole recursion stop
- **Two levels of butterflies**: one butterfly merge at the N=2 level (giving E and O), and one more at the N=4 level (merging E and O into the final result)

In [ ]:
import numpy as np
import time
from aurora.audio.transforms import fft as aurora_fft

## The big-picture map: the N=8 recursive divide-and-conquer tree

Look at the shape of the tree before you write the recursion. Going down is **splitting** (even/odd); going up is **butterfly merging**. The 3 levels correspond to `log₂8=3` levels of recursion, each with N/2=4 butterfly multiplications, 12 in total — the naive DFT needs 64.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(10, 4.2))

COLORS = {8: '#3498db', 4: '#e74c3c', 2: '#2ecc71', 1: '#95a5a6'}
LABELS = {8: 'my_fft(8)', 4: 'my_fft(4)', 2: 'my_fft(2)', 1: 'N=1\nreturn x'}
ys = [3.5, 2.4, 1.3, 0.2]   # y-centers per level
bh = 0.65                     # block height

def box(ax, cx, cy, w, n):
    ax.add_patch(plt.Rectangle((cx - w/2, cy - bh/2), w, bh,
                                fc=COLORS[n], ec='white', lw=2, alpha=0.88, zorder=3))
    ax.text(cx, cy, LABELS[n], ha='center', va='center',
            fontsize=8.5, fontweight='bold', color='white', zorder=4)

# Boxes at each level
box(ax, 5.0, ys[0], 9.0, 8)
for cx in [2.5, 7.5]:
    box(ax, cx, ys[1], 4.1, 4)
for cx in [1.25, 3.75, 6.25, 8.75]:
    box(ax, cx, ys[2], 1.75, 2)
for cx in np.linspace(0.6, 9.4, 8):
    box(ax, cx, ys[3], 0.68, 1)

# Arrows (split direction, going down)
kw = dict(arrowstyle='->', color='#7f8c8d', lw=1.2)
for cx in [2.5, 7.5]:
    ax.annotate('', xy=(cx, ys[1] + bh/2 + 0.04),
                xytext=(5.0, ys[0] - bh/2 - 0.04), arrowprops=kw, zorder=2)
for parent, children in [(2.5, [1.25, 3.75]), (7.5, [6.25, 8.75])]:
    for cx in children:
        ax.annotate('', xy=(cx, ys[2] + bh/2 + 0.04),
                    xytext=(parent, ys[1] - bh/2 - 0.04), arrowprops=kw, zorder=2)
child_xs = np.linspace(0.6, 9.4, 8)
for i, parent in enumerate([1.25, 3.75, 6.25, 8.75]):
    for cx in child_xs[i * 2:(i + 1) * 2]:
        ax.annotate('', xy=(cx, ys[3] + bh/2 + 0.04),
                    xytext=(parent, ys[2] - bh/2 - 0.04), arrowprops=kw, zorder=2)

# Right-side annotations
for y, lbl in zip(ys, ['level 0: 1×FFT(8)', 'level 1: 2×FFT(4)  even/odd split',
                        'level 2: 4×FFT(2)', 'level 3: 8×FFT(1)  base case']):
    ax.text(10.2, y, lbl, ha='left', va='center', fontsize=8.2, color='#2c3e50')

ax.set_xlim(-0.2, 13.5)
ax.set_ylim(-0.2, 4.2)
ax.axis('off')
ax.set_title('N=8  my_fft(x) recursive divide-and-conquer tree\n'
             'split down (even/odd) → N=1 base case → butterfly-merge back up',
             fontsize=10.5, fontweight='bold', pad=4)
ax.legend(handles=[mpatches.Patch(fc=COLORS[n], ec='white', lw=1.5, alpha=0.88,
                                   label=f'FFT(N={n})') for n in [8, 4, 2, 1]],
          loc='lower right', fontsize=8.5, framealpha=0.9)
plt.tight_layout()
plt.show()

## 1. The recursive structure: butterfly merging

The core Cooley–Tukey identity:

```
X[k]     = E[k] + exp(-2πi·k/N) · O[k]      (k = 0..N/2-1)
X[k+N/2] = E[k] - exp(-2πi·k/N) · O[k]      (k = 0..N/2-1)
```

where `E = fft(x[::2])` (the even indices) and `O = fft(x[1::2])` (the odd indices).

### Why does the second line use a minus sign?

It isn't an arbitrary choice — it **falls out automatically from the periodicity in the DFT definition** (see the derivation section above).

In short:
- the first half, $X[k]$, uses $E[k] + W^k \cdot O[k]$
- the second half, $X[k+N/2]$, uses $E[k] - W^k \cdot O[k]$

The minus sign appears because once you add $N/2$ to $k$ in the exponent of $e^{-2\pi i (k+N/2) n / N}$, every **odd term $n=2m+1$** picks up an extra factor of $e^{-\pi i} = -1$. That's why "no extra multiplication is needed" — the minus sign is "free", handed to us by the DFT formula itself.

Multiplying by the twiddle factor `exp(-2πi·k/N)` and then adding/subtracting is the "butterfly" operation.
Base case: when `N==1`, the DFT is just the value itself, so return it directly.

In [ ]:
# Demo: the N=2 DFT by hand vs the FFT formula
x2 = np.array([3.0, 7.0])
# DFT definition: X[k] = sum_n x[n]*exp(-2pi*i*k*n/N)
X2_def = np.array([
    x2[0] + x2[1],           # k=0: exp(0)=1
    x2[0] - x2[1],           # k=1: exp(-pi*i)=-1
], dtype=complex)
X2_np = np.fft.fft(x2)
print('by hand X[0]:', X2_def[0], '  numpy:', X2_np[0])
print('by hand X[1]:', X2_def[1], '  numpy:', X2_np[1])
print('error:', np.max(np.abs(X2_def - X2_np)))

## 2. It must be a power of 2; for other lengths, zero-pad

Cooley–Tukey halves N at every step, so it needs N to be a power of 2 (`N = 2^k`). This is **a requirement of the algorithm's design**, not "a programming limitation".

Why? Because every level of the recursion must split **exactly in half**: N → N/2 → N/4 → ... → 1. If N isn't a power of 2 (say N=5), you can't keep halving it (5 → 2.5 ✗).

### What about signals whose length isn't a power of 2?

The standard approach is to find the smallest `N_padded = 2^ceil(log2(L))`, pad zeros onto the end up to N_padded, and get an N_padded-point spectrum.
```
L = 5  →  N_padded = 2^ceil(log2(5)) = 2^3 = 8
L = 100  →  N_padded = 2^ceil(log2(100)) = 2^7 = 128
```

Note: zero-padding **does not increase** the physical frequency resolution (that is set by the **duration** of the original signal); it only makes the frequency grid denser, so the spectral lines look smoother.

### Example: a real signal

Say you have 5 seconds of audio at a 16 kHz sample rate, 80000 samples in all. Zero-pad it to 2^17 = 131072, and the frequency resolution is still `16000 Hz / 80000 = 0.2 Hz` (set by the original 5-second duration), not `16000 Hz / 131072`. Zero-padding only spreads the frequency grid more densely; it doesn't change the "physical resolution".

In [ ]:
# Demo: padding N=5 to 8, and what zero-padding does to frequency resolution
import numpy as np
import matplotlib.pyplot as plt

# Build a simple signal: a 440 Hz + 880 Hz mix (two notes)
sr = 8000  # sample rate 8 kHz
duration = 1.0  # 1 second
t = np.linspace(0, duration, int(sr * duration), endpoint=False)
f1, f2 = 440, 880
x_long = np.sin(2 * np.pi * f1 * t) + np.sin(2 * np.pi * f2 * t)

print("Example 1: zero-padding any length up to a power of 2")
print("-" * 60)
for L in [5, 100, 1000]:
    x = x_long[:L]
    N_padded = 2 ** int(np.ceil(np.log2(L)))
    x_pad = np.pad(x, (0, N_padded - L))
    print(f"  original length {L:4d} → padded to {N_padded:5d}")

print()
print("Example 2: zero-padding doesn't change the 'physical frequency resolution'")
print("-" * 60)
# Take the first 4000 samples (0.5 s) of the real 1-second signal
x_0p5s = x_long[:4000]  # 0.5 s, 8000 Hz × 0.5 = 4000 samples
print(f"signal duration: 0.5 s, sample rate 8000 Hz")
print(f"original sample count: {len(x_0p5s)}")

# FFT without padding (pretend 4000 is a power of 2... it isn't, but this shows the principle)
N_pad_to = 2 ** int(np.ceil(np.log2(len(x_0p5s))))
print(f"padded to 2^{int(np.log2(N_pad_to))} = {N_pad_to}")

x_padded = np.pad(x_0p5s, (0, N_pad_to - len(x_0p5s)))
X_padded = np.fft.fft(x_padded)
freq_resolution = sr / len(x_0p5s)  # physical resolution: set by the original duration
freq_axis = np.fft.fftfreq(N_pad_to, 1/sr)

print(f"physical frequency resolution: {sr} Hz / {len(x_0p5s)} = {freq_resolution:.2f} Hz")
print(f"  (set by the original 0.5 s duration; it doesn't change)")
print(f"frequency grid points after padding: {N_pad_to}")
print(f"  (a denser grid, but the ability to separate the 440/880 Hz signals is unchanged)")
print()

# Plot the comparison
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
# magnitude spectrum (up to 1000 Hz)
ax = axes[0]
freq_mask = freq_axis <= 1000
ax.plot(freq_axis[freq_mask], np.abs(X_padded[freq_mask]) / len(x_0p5s),
        linewidth=1, alpha=0.7, label='spectrum after padding (denser grid)')
ax.axvline(440, color='red', linestyle='--', alpha=0.5, label='440 Hz')
ax.axvline(880, color='green', linestyle='--', alpha=0.5, label='880 Hz')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude')
ax.set_title('Zero-padding makes the frequency grid denser, but 440/880 Hz resolution is unchanged (still set by the 0.5 s duration)')
ax.legend()
ax.grid(True, alpha=0.3)

# before vs after padding, sketched
ax = axes[1]
x_vis = x_0p5s[:200]  # visualize the first 200 samples
x_pad_vis = x_padded[:200]
t_vis = np.arange(len(x_vis)) / sr
ax.plot(t_vis, x_vis, 'o-', markersize=3, alpha=0.6, label='original signal (first 200 samples)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('Zero-padding leaves the original signal alone (it only appends zeros at the end)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Summary: zero-padding only changes the **density** of the frequency grid (from a 8000/4000=2 Hz grid")
print("     to a denser 8000/4096≈1.95 Hz grid); it doesn't change the **physical resolution**.")


## 3. Numerical error: floating-point accumulation < 1e-10

A recursive FFT has `log₂N` levels, each doing floating-point multiplications; the error builds up linearly with the number of levels, on the order of `N·log₂N·ε_machine`.
For `N=1024` in double precision, that's about `1024·10·2.2e-16 ≈ 2e-12`, far below `1e-10`.
The tool for checking it: `np.testing.assert_allclose(my_result, reference, atol=1e-10)`, which prints where the largest deviation is if it fails.

```
# atol is the absolute tolerance, rtol the relative one; for complex spectra set both
np.testing.assert_allclose(A, B, rtol=1e-9, atol=1e-10)
```

In [ ]:
# Demo: measure the floating-point error for different N (naive DFT vs numpy.fft, to show the size of the error)
print("     N  max_abs_err (naive_dft vs numpy.fft)")
for N in [4, 16, 64, 256, 1024, 4096]:
    x = np.random.randn(N)
    ref = np.fft.fft(x)
    n_arr = np.arange(N)
    k_arr = n_arr.reshape((N, 1))
    M_naive = np.exp(-2j * np.pi * k_arr * n_arr / N)
    naive = M_naive @ x.astype(complex)
    err = np.max(np.abs(naive - ref))  # difference between the naive O(N²) DFT and numpy
    print(f"{N:>6}  {err:.2e}")
print("\n(These are the naive-DFT vs numpy.fft errors; once my_fft is implemented, the check cell below verifies that its own error is < 1e-10)")


## Visualizing the call stack: in what order does `my_fft` actually run? (understanding the program's recursion, not the FFT maths)

The divide-and-conquer tree above shows the "structure". Before you write the `return`, you also need to see the program's **real order of execution**. Recursion is **depth first**: `my_fft(x[::2])` drills all the way to the bottom, and only after it returns does `my_fft(x[1::2])` get its turn. Taking N=4 as the example, here is every function entry and exit laid out in time order:

```text
my_fft([a,b,c,d])              entered, N=4
├─ E = my_fft([a,c])           even half first, enter N=2
│   ├─ my_fft([a]) → [a]       bottom: N==1, return directly (base case)
│   ├─ my_fft([c]) → [c]       bottom: N==1, return directly
│   └─ butterfly merge → E      only once both single points are back do they merge into E
├─ O = my_fft([b,d])           only after the even half fully returns does the odd half start
│   ├─ my_fft([b]) → [b]
│   ├─ my_fft([d]) → [d]
│   └─ butterfly merge → O
└─ butterfly merge(E, O) → X    with E and O both in place, one last merge gives the answer
```

How to read it: **every vertical line is a `my_fft` that "hasn't returned yet and is still sitting on the call stack"**. The deepest point of the stack is always a call of length 1 — it doesn't recurse any further, it returns straight away, and it's where the whole tree starts "reeling back in". So when you write the code, the line `if N == 1: return ...` decides whether this picture can close up properly; leave it out and the recursion drills down forever (`RecursionError`).

## 4. ✏️ Implement `my_fft(x)` (recursive Cooley–Tukey)

**Five-step hint**:

1. Base case: when the length is 1, return the input directly (as complex)
2. Even/odd split: separate the even positions and the odd positions into two subproblems
3. Twiddle factors: `twiddle` only takes care of the phase steps around one circle
4. Butterfly merge: combine `E` and `O` into the top / bottom halves of the spectrum
5. Concatenate and return: top half first, then bottom half

**Accuracy requirement**: `np.allclose(my_fft(x), np.fft.fft(x), atol=1e-10)` passes

**Stuck? Go back to**: L38's butterfly diagram, the divide-and-conquer trees in L38/L39, or `solutions/L39_fft_implement_solutions.md`


In [ ]:
def my_fft(x: np.ndarray) -> np.ndarray:
    """Recursive Cooley-Tukey radix-2 FFT. The length of x must be a power of 2."""
    N = len(x)
    # ✏️ TODO 1: base case — return directly when N==1
    raise NotImplementedError("TODO 1-5: implement the full recursive Cooley-Tukey FFT (base case / even-odd split / twiddle factors / butterfly merge / concatenate)")

    # ✏️ TODO 2: recursively split into even/odd indices
    E = ...  # my_fft(x[::2])
    O = ...  # my_fft(x[1::2])

    # ✏️ TODO 3: twiddle factors twiddle = exp(-2pi*i * k / N), k = 0..N/2-1
    k = np.arange(N // 2)
    twiddle = ...

    # ✏️ TODO 4: butterfly merge
    top    = ...  # E + twiddle * O
    bottom = ...  # E - twiddle * O

    # ✏️ TODO 5: concatenate and return
    return ...

In [ ]:
# ── Correctness check ──────────────────────────────────────────────
import numpy as np
x_demo = np.array([1, 2, 3, 4, 3, 2, 1, 0], dtype=float)
try:
    out_demo = my_fft(x_demo)
    if out_demo is None or out_demo is ...:
        print("⬜ Fill in the TODOs in my_fft first")
    else:
        ref_demo = np.fft.fft(x_demo)
        print('max error:', np.max(np.abs(out_demo - ref_demo)))
        all_pass = True
        for N in [4, 8, 16, 64, 256]:
            x = np.random.randn(N)
            if not np.allclose(my_fft(x), np.fft.fft(x), atol=1e-10):
                print(f'✗ N={N} failed'); all_pass = False
        if all_pass:
            print('✅ all N in [4,8,16,64,256] pass, error < 1e-10')
except (NotImplementedError, TypeError):
    print("⬜ Fill in the TODOs in my_fft first, then run the check cell")
except TypeError:
    print("⬜ Replace the ... placeholders first, then run the check cell")
except RecursionError:
    print("⬜ my_fft recursion error: check the base case (the return when N==1)")


In [ ]:
# ── Aurora connection: check that my_fft and aurora_fft agree numerically ─────────────────────────────
# aurora_fft was imported in the setup cell above (cell 12); here we do the final comparison
try:
    x_check = np.array([1, 2, 3, 4, 3, 2, 1, 0], dtype=float)
    out_my = my_fft(x_check)
    if out_my is None or out_my is ...:
        print("□ Implement my_fft first, then run this cell")
    else:
        out_aurora = aurora_fft(x_check)
        max_diff = float(np.max(np.abs(out_my - out_aurora)))
        np.testing.assert_allclose(out_my, out_aurora, atol=1e-10,
            err_msg="my_fft and aurora_fft disagree; check your butterfly-merge logic")
        print(f"✅ my_fft ↔ aurora_fft max difference: {max_diff:.2e}  (< 1e-10)")
        print("   numerically equivalent, by different algorithmic routes (recursive vs iterative Cooley-Tukey)")
except (NotImplementedError, TypeError):
    print("□ Implement my_fft first, then run this cell")
except TypeError:
    print("□ Replace the ... placeholders first, then run it")
except RecursionError:
    print("□ my_fft recursion error: check the base case (the return when N==1)")


## Extra · non-powers of 2, zero-padding, and the STFT calling convention

This lesson's `my_fft(x)` only has to handle power-of-2 lengths; that keeps the recursive divide and conquer clear.

The production code has the same boundary:

- `aurora.audio.transforms.fft(x)`: the input length must be a power of 2; otherwise it raises `ValueError`
- `aurora.audio.transforms.next_power_of_two(n)`: returns the smallest power of 2 that is no less than `n`
- `aurora.audio.stft.stft(...)`: before each frame goes into `rfft/fft`, it guards `n_fft` to a power-of-2 length

So you can remember the convention as:

```text
plain FFT: it's your job to hand it a power-of-2 length
STFT pipeline: each frame gets zero-padded as a guard, by n_fft/next_power_of_two
```

This comes up again when you write the STFT by hand in L44: `win_len` / `n_fft` are usually powers of 2 like 256, 512 or 1024.


In [ ]:
import numpy as np
from aurora.audio.transforms import fft, next_power_of_two

x5 = np.ones(5)
try:
    fft(x5)
except ValueError as exc:
    print("Calling fft directly on a non-power-of-2 length fails:", str(exc).split(".")[0])

target = next_power_of_two(len(x5))
x_pad = np.pad(x5, (0, target - len(x5)))
X_pad = fft(x_pad)

assert target == 8
assert x_pad.shape == (8,)
assert X_pad.shape == (8,)
print(f"✅ len=5 → runs once padded to {target}; this is exactly the boundary awareness L44/STFT will need")


## 5. Parameter experiment: comparing speeds

- `N = 512`: naive DFT (O(N²)) vs `my_fft` (O(N log N)) vs `np.fft.fft` (a C extension)
- What to expect: in operation counts, `my_fft` beats the naive DFT by a factor of `N²/((N/2)·log₂N) = 2N/log₂N ≈ 114` at `N = 512`; the measured gap is much smaller, because `naive_dft` here is a single vectorized NumPy matrix product while `my_fft` makes hundreds of Python calls. `np.fft.fft` is far faster than `my_fft` again (C extension + SIMD), often by a factor of 100 or more
- Change `N` and watch: the ratios grow as `N` grows (the log-sized gain becomes more pronounced)

In [ ]:
def naive_dft(x: np.ndarray) -> np.ndarray:
    """Brute-force O(N²) DFT, used as the speed baseline."""
    N = len(x)
    n = np.arange(N)
    k = n.reshape((N, 1))
    M = np.exp(-2j * np.pi * k * n / N)
    return M @ x.astype(complex)

N = 512
x_bench = np.random.randn(N)

reps = 20
t0 = time.perf_counter()
for _ in range(reps): naive_dft(x_bench)
t_naive = (time.perf_counter() - t0) / reps * 1000

t0 = time.perf_counter()
for _ in range(reps): my_fft(x_bench)
t_my = (time.perf_counter() - t0) / reps * 1000

t0 = time.perf_counter()
for _ in range(reps): np.fft.fft(x_bench)
t_np = (time.perf_counter() - t0) / reps * 1000

print(f'N={N}  (average of {reps} runs, in ms)')
print(f'  naive_dft : {t_naive:8.3f} ms')
print(f'  my_fft    : {t_my:8.3f} ms   ({t_naive/t_my:.1f}× faster than naive)')
print(f'  np.fft.fft: {t_np:8.3f} ms   ({t_my/t_np:.1f}× faster than my_fft)')
print(f'\nTheoretical op-count speedup naive→FFT is 2N/log₂N (about 114× at N=512); the number of FFT stages is log₂({N}) = {np.log2(N):.1f}')

---
→ **Once `my_fft` is implemented, open [L42 · The FFT drawn](L42_visual_fft.ipynb), compare the butterfly diagram with the spectra of different kinds of signals, check that the output matches your intuition, then come back.**

## 🎯 The payoff later

The recursive `my_fft` you write yourself today will come back to you again and again in **L41 / L43–L44 / L60+** — L41 wraps it into a windowed FFT; **the STFT in L43–L44 calls the very FFT you wrote today on every frame of the signal, over and over**; and the whole STFT → mel → MFCC feature pipeline (which in the end feeds the neural networks later on) stands on top of this one recursion. Once you really understand "how much one FFT call costs", you'll see why real-time audio has to budget carefully for every single frame.

## Wrap-up

`my_fft(x)` outputs N complex spectral coefficients through recursive butterfly operations, with the error held within `1e-10`. It is numerically equivalent to `aurora.audio.transforms.fft`, but the algorithmic route is different: the production version uses an **iterative** Cooley–Tukey (a bit-reversal permutation + non-recursive butterflies), which L38's appendices walk through; both can correctly drive the inner computation of the STFT. Next lesson, L40, analyzes the magnitude and phase spectra of the complex spectrum, and checks how frequency resolution relates to signal length.

---
⬇️ **Completion check**: you've read the wrap-up; please finish the whiteboard challenge below before ticking the self-review.


## ✏️ Whiteboard challenge: the FFT by hand, and its recursive structure (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: what is the base case of the `my_fft` recursion?
(For N=?, it returns directly — and what does it return?)

**Question 2**: for an N=8 input, how many levels does the recursion tree have? How many butterfly operations are there per level?

**Question 3**: for `x = [1, 0, -1, 0]`, work out X[1] by hand with Cooley–Tukey divide and conquer:
- x_even = x[0,2] = [1, -1], E = DFT([1,-1]) = ?
- x_odd  = x[1,3] = [0, 0], O = DFT([0, 0])  = ?
- W_4^1 = e^{-πi/2} = ?
- X[1] = E[1] + W_4^1 · O[1] = ?

**Question 4**: what is the result of `my_fft(np.array([1,1,1,1]))`?
(Hint: the DFT of a constant signal is non-zero only at k=0)

**Question 5**: why does `my_fft` need the input length to be a power of 2? How do you handle it when it isn't?

Once you've done the derivations, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: base case N=1
x1 = np.array([3.7])
X1_ref = np.fft.fft(x1)
try:
    X1 = my_fft(x1)
    assert np.allclose(X1, X1_ref, atol=1e-12)
    print(f"Q1 ✅  N=1 base case: my_fft([3.7])={X1} (returns the input itself, as a complex number)")
except (NotImplementedError, TypeError):
    print("⬜ Q1: implement my_fft() first, then run the answer-check cell")

# Q2: the N=8 tree structure
N8 = 8
n_layers = int(np.log2(N8))
butterflies_per_layer = N8 // 2
total_butterflies = butterflies_per_layer * n_layers
assert n_layers == 3 and butterflies_per_layer == 4 and total_butterflies == 12
print(f"Q2 ✅  N=8: {n_layers} levels, {butterflies_per_layer} butterflies per level, {total_butterflies} butterflies in all")

# Q3: x=[1,0,-1,0], X[1] by hand
x4 = np.array([1.0, 0.0, -1.0, 0.0])
X4_ref = np.fft.fft(x4)
# E = DFT([1,-1]) = [0, 2]
E4 = np.fft.fft(np.array([1.0, -1.0]))
O4 = np.fft.fft(np.array([0.0, 0.0]))
W41 = np.exp(-2j * np.pi * 1 / 4)
X1_calc = E4[1] + W41 * O4[1]
assert np.isclose(X1_calc, X4_ref[1], atol=1e-12)
print(f"Q3 ✅  E[1]={E4[1]:.4f}, O[1]={O4[1]:.4f}, W_4^1={W41:.4f}")
print(f"      X[1] = {E4[1]:.4f} + {W41:.4f}×{O4[1]:.4f} = {X1_calc:.4f}")
try:
    X4_mine = my_fft(x4)
    assert np.allclose(X4_mine, X4_ref, atol=1e-9)
    print(f"      my_fft check: {np.round(X4_mine, 4)}  ✅")
except (NotImplementedError, TypeError):
    print("      ⬜ implement my_fft() first, then run the answer-check cell")

# Q4: x=[1,1,1,1]
x_dc = np.array([1.0, 1.0, 1.0, 1.0])
X_dc_ref = np.fft.fft(x_dc)
assert np.isclose(X_dc_ref[0], 4+0j, atol=1e-12)
assert np.allclose(X_dc_ref[1:], 0+0j, atol=1e-12)
print(f"Q4 ✅  DFT([1,1,1,1]) = {np.round(X_dc_ref, 4)} (4 at k=0 only, 0 everywhere else)")

# Q5: powers of 2 — other lengths need zero-padding
x5 = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
N_padded = 2 ** int(np.ceil(np.log2(len(x5))))
assert N_padded == 8
x5_padded = np.pad(x5, (0, N_padded - len(x5)))
X5_padded = np.fft.fft(x5_padded)
print(f"Q5 ✅  N=5 isn't a power of 2, pad to N={N_padded}; spectrum length={len(X5_padded)} (an interpolated frequency axis)")
print("\n🎉 FFT-by-hand challenge passed! The recursive Cooley-Tukey structure is now yours.")

In [ ]:
# ✏️ Self-review for this lesson
l39_review = {
    "cooley_tukey_recursion":  None,  # remember the recursive structure: base N=1, split even/odd, butterfly? True/False
    "my_fft_implemented":      None,  # my_fft implemented and passing the atol=1e-10 check? True/False
    "error_bound_understood":  None,  # understand where the < 1e-10 floating-point error comes from (accumulating over log N levels)? True/False
    "speedup_quantified":      None,  # know that at N=1024 the FFT is about 205 times faster than the DFT? True/False
    "whiteboard_passed":       None,  # whiteboard challenge derived on paper? True/False
    "stft_padding_contract":   None,  # understand that fft needs a power of 2 and the STFT zero-pads as a guard? True/False
}

unfilled = [k for k, v in l39_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l39_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L39 fully cleared! On to L40: using the spectrum to explain real signals')

---

→ **Next lesson** · [L40 · Spectrum analysis in practice](L40_spectrum.ipynb)

> Next lesson covers **spectrum analysis in practice**: magnitude spectrum / phase spectrum / frequency resolution, with a 440 Hz + 880 Hz mix.